In [2]:
import pandas as pd

df = pd.read_csv(
    "/kaggle/input/datasets/muqaddasejaz/fake-reviews-dataset/fake reviews dataset.csv"
)

print(df.shape)
print(df.columns)
df.head()

(40432, 4)
Index(['category', 'rating', 'label', 'text_'], dtype='object')


,category,rating,label,text_
0,Home_and_Kitchen_5,5.0,CG,"Love this! Well made, sturdy, and very comfor..."
1,Home_and_Kitchen_5,5.0,CG,"love it, a great upgrade from the original. I..."
2,Home_and_Kitchen_5,5.0,CG,This pillow saved my back. I love the look and...
3,Home_and_Kitchen_5,1.0,CG,"Missing information on how to use it, but it i..."
4,Home_and_Kitchen_5,5.0,CG,Very nice set. Good quality. We have had the s...


In [3]:
print(df['label'].value_counts())
print(df['label'].unique())

label
CG    20216
OR    20216
Name: count, dtype: int64
['CG' 'OR']


In [4]:
df['label'] = df['label'].map({
    'CG': 1,   # fake
    'OR': 0    # genuine
})

print(df['label'].value_counts())
print(df['label'].isna().sum())

label
1    20216
0    20216
Name: count, dtype: int64
0


In [5]:
import re

def clean_text(text):
    text = str(text).lower()

    text = re.sub(r'http\S+', ' ', text)

    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    text = re.sub(r'\s+', ' ', text)

    return text.strip()

df['clean_review'] = df['text_'].apply(clean_text)

df[['text_','clean_review']].head()

,text_,clean_review
0,"Love this! Well made, sturdy, and very comfor...",love this well made sturdy and very comfortabl...
1,"love it, a great upgrade from the original. I...",love it a great upgrade from the original i ve...
2,This pillow saved my back. I love the look and...,this pillow saved my back i love the look and ...
3,"Missing information on how to use it, but it i...",missing information on how to use it but it is...
4,Very nice set. Good quality. We have had the s...,very nice set good quality we have had the set...


In [6]:
from sklearn.model_selection import train_test_split

X = df['clean_review']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(32345,)
(8087,)


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)

X_train_tfidf = vectorizer.fit_transform(X_train)

X_test_tfidf = vectorizer.transform(X_test)

print(X_train_tfidf.shape)

(32345, 5000)


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)

X_train_tfidf = vectorizer.fit_transform(X_train)

X_test_tfidf = vectorizer.transform(X_test)

print(X_train_tfidf.shape)

(32345, 5000)


In [9]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [10]:
from sklearn.metrics import accuracy_score, classification_report

pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, pred))

print("\nClassification Report:\n")

print(classification_report(y_test, pred))

Accuracy: 0.8654630889081242

Classification Report:

              precision    recall  f1-score   support

           0       0.86      0.87      0.87      4044
           1       0.87      0.86      0.86      4043

    accuracy                           0.87      8087
   macro avg       0.87      0.87      0.87      8087
weighted avg       0.87      0.87      0.87      8087



In [11]:
review = """
Amazing product. Best purchase ever.
Highly recommended.
"""

review = clean_text(review)

vector = vectorizer.transform([review])

prediction = model.predict(vector)[0]

probability = model.predict_proba(vector)[0][1]

trust_score = int((1 - probability) * 100)

if prediction == 1:
    print("Prediction : Fake Review")
else:
    print("Prediction : Genuine Review")

print("Trust Score :", trust_score, "/100")

Prediction : Genuine Review
Trust Score : 79 /100


In [12]:
def analyze_review(review):

    review = clean_text(review)

    vector = vectorizer.transform([review])

    prediction = model.predict(vector)[0]

    fake_probability = model.predict_proba(vector)[0][1]

    trust_score = int((1 - fake_probability) * 100)

    if prediction == 1:
        label = "Fake Review"
    else:
        label = "Genuine Review"

    print("Prediction:", label)
    print("Trust Score:", trust_score, "/100")


analyze_review(
    "Amazing product. Best purchase ever. Highly recommended."
)

Prediction: Genuine Review
Trust Score: 79 /100


In [13]:
feature_names = vectorizer.get_feature_names_out()

coef = model.coef_[0]

top_fake_words = sorted(
    zip(feature_names, coef),
    key=lambda x: x[1],
    reverse=True
)[:20]

print("Top words indicating Fake Reviews:\n")

for word, score in top_fake_words:
    print(word, round(score,3))

Top words indicating Fake Reviews:

reason 6.978
problem 6.861
couple 5.456
wide 5.421
replace 5.14
admit 5.097
downside 4.823
materials 4.738
plastic 4.532
lot 4.373
developed 4.346
fan 3.783
strong 3.714
bit 3.677
pieces 3.659
fact 3.585
characters 3.377
small 3.357
acting 3.268
trs 3.267
